In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime as dt

In [2]:
# Binance Transactions
df = pd.read_csv('bin_trans.csv', delimiter=';')
# extendable by user input
operations_dep = ['Deposit','Buy Crypto With Fiat']
df_input = df[df.Operation.isin(operations_dep)]

# Data wrangling
df_input['Coin'] = df_input['Coin'].apply(lambda x: x[:3])
df_input['Change'] = round(df_input['Change'].astype(float),2)
df_input.rename(columns={'Change':'Value'}, inplace=True)
df_input.reset_index(drop=True, inplace=True)
df_input['UTC_Time'] = df_input['UTC_Time'].str.split(' ').str[0]
df_input['UTC_Time'] = pd.to_datetime(df_input['UTC_Time'])
df_input = df_input.iloc[:,1:-1]
df_input['FX_Change'] = 1
for ind in range(0,len(df_input)):
    curr_ = df_input.iloc[ind,3] 
    if curr_ != "PLN":
        df_input['FX_Change'][ind] = round(yf.download(f'{curr_}PLN=X',start=df_input.iloc[ind,0], end=df_input.iloc[ind,0] + pd.DateOffset(days=1)).iloc[0,0],2)
df_input['Value_PLN'] = round(df_input.Value * df_input.FX_Change,2)

# sum of deposits
df_input.Value_PLN.sum()

df_trans = df[~df['Remark'].isin(["Binance Earn", "Binance Launchpool"])]
df_trans['Change'] = df_trans['Change'].astype(float)

df_trans = pd.merge(df_trans[(df_trans['Operation'] == "Binance Convert") & (df_trans["Change"]>0)].loc[:,['UTC_Time','Operation','Coin','Change']],
         df_trans[(df_trans['Operation'] == "Binance Convert") & (df_trans["Change"]<0)].loc[:,['UTC_Time','Operation','Coin','Change']],
         on = 'UTC_Time').drop_duplicates(keep='first')

/var/folders/x8/q__bzqys7yg57g9twpxbqxqm0000gn/T/ipykernel_1270/940259602.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_input['Coin'] = df_input['Coin'].apply(lambda x: x[:3])
/var/folders/x8/q__bzqys7yg57g9twpxbqxqm0000gn/T/ipykernel_1270/940259602.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_input['Change'] = round(df_input['Change'].astype(float),2)
/var/folders/x8/q__bzqys7yg57g9twpxbqxqm0000gn/T/ipykernel_1270/940259602.py:10: SettingWithCopyWarning: 
A value is trying to be set on a

In [4]:
df['Change'] = df['Change'].astype(str).str.strip(' ').str.replace(',','.').astype(float)

In [5]:
df_test = pd.merge(df[(df['Change'] > 0 )].loc[:,['UTC_Time','Operation','Coin','Change']],
         df[(df['Change'] < 0 )].loc[:,['UTC_Time','Operation','Coin','Change']],
         on = 'UTC_Time', how ='left').drop_duplicates(keep='first')

In [6]:
df_test = df_test.drop_duplicates(
    subset=['UTC_Time', 'Operation_x', 'Coin_x', 'Change_x'], 
    keep='first'
).reset_index(drop=True)

In [11]:
df = pd.read_csv('bin_trans.csv', delimiter=';')
df['Change'] = df['Change'].astype(str).str.strip(' ').str.replace(',','.').astype(float)
df

,User_ID,UTC_Time,Account,Operation,Coin,Change,Remark
0,71732789,17/02/2021 11:48,Spot,Deposit,EUR,2.400000e+02,NaN
1,71732789,18/02/2021 11:05,Spot,Sell,EUR,-4.989270e+01,NaN
2,71732789,18/02/2021 11:05,Spot,Buy,BTC,1.163000e-03,NaN
3,71732789,18/02/2021 11:06,Spot,Sell,EUR,-4.999009e+01,NaN
4,71732789,18/02/2021 11:06,Spot,Buy,ETH,3.147000e-02,NaN
...,...,...,...,...,...,...,...
4467,71732789,11/09/2025 04:44,Spot,HODLer Airdrops Distribution,HOLO,5.001740e+00,Binance Launchpool
4468,71732789,12/09/2025 04:14,Spot,Simple Earn Flexible Interest,BTC,0.000000e+00,Binance Earn
4469,71732789,12/09/2025 04:20,Spot,Simple Earn Flexible Interest,ETH,4.000000e-07,Binance Earn
4470,71732789,12/09/2025 04:22,Spot,Simple Earn Flexible Interest,USDC,1.484872e-01,Binance Earn


In [12]:
df[df['Coin']=='LUNA']

,User_ID,UTC_Time,Account,Operation,Coin,Change,Remark
99,71732789,09/04/2021 08:29,Spot,Buy,LUNA,3.80,NaN
134,71732789,21/04/2021 16:09,Spot,Buy,LUNA,8.10,NaN
208,71732789,30/10/2021 15:01,Spot,Transaction Sold,LUNA,-9.23,NaN
240,71732789,22/01/2022 10:02,Spot,Transaction Buy,LUNA,2.07,NaN
247,71732789,27/01/2022 21:54,Spot,Transaction Buy,LUNA,2.15,NaN
259,71732789,05/02/2022 02:59,Spot,Transaction Sold,LUNA,-6.89,NaN


In [17]:
df[(df['Coin']=='BNB')]['Operation'].unique()

array(['Buy', 'Fee', 'Launchpool Subscription/Redemption',
       'Launchpad Subscribe', 'Sell', 'Staking Purchase',
       'Staking Rewards', 'Staking Redemption', 'Transaction Fee',
       'Transaction Sold', 'Simple Earn Flexible Subscription',
       'Simple Earn Flexible Interest', 'Simple Earn Flexible Redemption',
       'Transaction Revenue', 'Binance Convert',
       'Simple Earn Locked Subscription', 'Simple Earn Locked Rewards',
       'Transfer Between Main and Funding Wallet'], dtype=object)

In [49]:
df_coins = pd.DataFrame(df[(df['Operation'].isin(['Asset Recovery','Deposit','Buy','Fee','Sell','Stacking Rewards','Transaction Fee','Transaction Sold','Simple Earn Flexible Interest','Transaction Revenue','Binance Convert','Simple Earn Locked Rewards','Transaction Spend','Transfer Between Main and Funding Wallet']))].groupby(by='Coin')['Change'].sum())

,Change
Coin,
1000CAT,-7.779933e+01
AAVE,6.830474e-18
ADA,1.672155e+02
AEVO,-2.202717e+00
ALICE,-1.011832e+01
...,...
USDC,1.350067e+03
USDT,-8.613548e+02
USUAL,-1.442056e+01


In [32]:
df[(df['Coin']=='EUR')]

,User_ID,UTC_Time,Account,Operation,Coin,Change,Remark
0,71732789,17/02/2021 11:48,Spot,Deposit,EUR,240.000000,NaN
1,71732789,18/02/2021 11:05,Spot,Sell,EUR,-49.892700,NaN
3,71732789,18/02/2021 11:06,Spot,Sell,EUR,-49.990095,NaN
6,71732789,20/02/2021 11:06,Spot,Sell,EUR,-49.860188,NaN
11,71732789,25/02/2021 22:34,Spot,Sell,EUR,-49.695800,NaN
13,71732789,28/02/2021 11:52,Spot,Sell,EUR,-40.560000,NaN
18,71732789,01/03/2021 11:57,Spot,Deposit,EUR,260.000000,NaN
19,71732789,01/03/2021 12:47,Spot,Sell,EUR,-60.000000,NaN
28,71732789,03/03/2021 20:33,Spot,Sell,EUR,-40.000000,NaN
30,71732789,03/03/2021 20:36,Spot,Sell,EUR,-160.000000,NaN


In [434]:
holdings = {}
holdings_in = {}
holdings_out = {}
for line in range(0,20):

    if df_test['Coin_x'][line] not in holdings:
        holdings.update({df_test['Coin_x'][line]:df_test['Change_x'][line]})
    else:
        holdings[df_test['Coin_x'][line]] = holdings[df_test['Coin_x'][line]] + df_test['Change_x'][line]
        
    if str(df_test['Coin_y'][line]) != 'nan':
        holdings[df_test['Coin_y'][line]] = holdings[df_test['Coin_y'][line]] + df_test['Change_y'][line]

    if df_test['Coin_x'][line] not in holdings_in:
        holdings_in.update({df_test['Coin_x'][line]:df_test['Change_x'][line]})
        holdings_out.update({df_test['Coin_x'][line]:0})
    else:
        holdings_in[df_test['Coin_x'][line]] = holdings_in[df_test['Coin_x'][line]] + df_test['Change_x'][line]

    if str(df_test['Coin_y'][line]) != 'nan':
        holdings_out[df_test['Coin_y'][line]] = holdings_out[df_test['Coin_y'][line]] + df_test['Change_y'][line]


In [435]:
holdings_in

{'EUR': np.float64(553.26),
 'BTC': np.float64(0.001163),
 'ETH': np.float64(0.03147),
 'DOGE': np.float64(1066.3),
 'FTM': np.float64(98.0),
 'ADA': np.float64(53.9),
 'USDT': np.float64(313.45422399999995),
 'FIO': np.float64(463.62),
 'BUSD': np.float64(48.18),
 'SUSHI': np.float64(2.755),
 'AAVE': np.float64(0.119),
 'RUNE': np.float64(8.35),
 'SNX': np.float64(2.085),
 'HEGIC': np.float64(184.31)}

In [502]:
df_test[(df_test['Coin_x']=='BNB') & (~df_test['Operation_x'].isin(['Launchpad Subscribe', 'Launchpool Subscription/Redemption']))]['Change_x'].sum() + df_test[(df_test['Coin_y']=='BNB') & (~df_test['Operation_y'].isin(['Launchpad Subscribe', 'Launchpool Subscription/Redemption']))]['Change_y'].sum()

np.float64(3.29870192)

In [507]:
df_test[(df_test['Coin_x']=='LUNA')]

,UTC_Time,Operation_x,Coin_x,Change_x,Operation_y,Coin_y,Change_y
39,09/04/2021 08:29,Buy,LUNA,3.80,Sell,BUSD,-63.30420
61,21/04/2021 16:09,Buy,LUNA,8.10,Sell,BNB,-0.19683
142,22/01/2022 10:02,Transaction Buy,LUNA,2.07,Transaction Spend,BUSD,-113.85000
148,27/01/2022 21:54,Transaction Buy,LUNA,2.15,Transaction Spend,BUSD,-112.87500


In [509]:
df[(df['Coin']=='LUNA')]

,User_ID,UTC_Time,Account,Operation,Coin,Change,Remark
99,71732789,09/04/2021 08:29,Spot,Buy,LUNA,3.80,NaN
134,71732789,21/04/2021 16:09,Spot,Buy,LUNA,8.10,NaN
208,71732789,30/10/2021 15:01,Spot,Transaction Sold,LUNA,-9.23,NaN
240,71732789,22/01/2022 10:02,Spot,Transaction Buy,LUNA,2.07,NaN
247,71732789,27/01/2022 21:54,Spot,Transaction Buy,LUNA,2.15,NaN
259,71732789,05/02/2022 02:59,Spot,Transaction Sold,LUNA,-6.89,NaN


In [503]:
df_test[(df_test['Coin_y']=='BNB') & (~df_test['Operation_y'].isin(['Launchpad Subscribe', 'Launchpool Subscription/Redemption']))]#['Change_y'].sum()

,UTC_Time,Operation_x,Coin_x,Change_x,Operation_y,Coin_y,Change_y
35,06/04/2021 12:20,Buy,BNB,0.149000,Fee,BNB,-0.000112
36,06/04/2021 22:57,Buy,BNB,0.145000,Fee,BNB,-0.000109
61,21/04/2021 16:09,Buy,LUNA,8.100000,Sell,BNB,-0.196830
97,08/05/2021 21:39,Buy,USDT,61.003361,Fee,BNB,-0.000071
99,17/05/2021 19:13,Buy,SXP,16.226000,Fee,BNB,-0.000089
121,30/10/2021 15:01,Transaction Revenue,EUR,346.125000,Transaction Fee,BNB,-0.000579
122,30/10/2021 15:02,Transaction Revenue,BUSD,3.105000,Transaction Sold,BNB,-0.006000
123,30/10/2021 15:02,Transaction Revenue,BUSD,112.815000,Transaction Sold,BNB,-0.006000
158,05/02/2022 02:59,Transaction Revenue,USDT,385.151000,Transaction Fee,BNB,-0.000725
163,09/03/2022 09:04,Transaction Buy,ADA,100.500000,Transaction Fee,BNB,-0.000162


In [484]:
df_test[df_test['Coin_y']=='BNB']

,UTC_Time,Operation_x,Coin_x,Change_x,Operation_y,Coin_y,Change_y
35,06/04/2021 12:20,Buy,BNB,0.149000,Fee,BNB,-0.000112
36,06/04/2021 22:57,Buy,BNB,0.145000,Fee,BNB,-0.000109
61,21/04/2021 16:09,Buy,LUNA,8.100000,Sell,BNB,-0.196830
97,08/05/2021 21:39,Buy,USDT,61.003361,Fee,BNB,-0.000071
99,17/05/2021 19:13,Buy,SXP,16.226000,Fee,BNB,-0.000089
121,30/10/2021 15:01,Transaction Revenue,EUR,346.125000,Transaction Fee,BNB,-0.000579
122,30/10/2021 15:02,Transaction Revenue,BUSD,3.105000,Transaction Sold,BNB,-0.006000
123,30/10/2021 15:02,Transaction Revenue,BUSD,112.815000,Transaction Sold,BNB,-0.006000
158,05/02/2022 02:59,Transaction Revenue,USDT,385.151000,Transaction Fee,BNB,-0.000725
163,09/03/2022 09:04,Transaction Buy,ADA,100.500000,Transaction Fee,BNB,-0.000162


In [455]:
df_test[df_test['Coin_x']=='BNB']['Change_x'].sum() + df_test[df_test['Coin_y']=='BNB']['Change_y'].sum()

np.float64(10.780456659999999)

In [482]:
df.iloc[224:265,:]

,User_ID,UTC_Time,Account,Operation,Coin,Change,Remark
224,71732789,10/01/2022 12:52,Spot,Transaction Fee,BNB,-0.000198,NaN
225,71732789,10/01/2022 14:20,Spot,Transaction Spend,BUSD,-100.000000,NaN
226,71732789,10/01/2022 14:20,Spot,Transaction Buy,ALICE,10.000000,NaN
227,71732789,11/01/2022 00:52,Spot,Staking Rewards,IOST,0.093737,NaN
228,71732789,12/01/2022 00:53,Spot,Staking Rewards,IOST,0.093737,NaN
229,71732789,13/01/2022 00:53,Spot,Staking Rewards,IOST,0.093737,NaN
230,71732789,14/01/2022 00:52,Spot,Staking Rewards,IOST,0.093737,NaN
231,71732789,15/01/2022 00:52,Spot,Staking Rewards,IOST,0.093737,NaN
232,71732789,16/01/2022 00:52,Spot,Staking Rewards,IOST,0.093737,NaN
233,71732789,17/01/2022 00:53,Spot,Staking Rewards,IOST,0.093737,NaN


In [477]:
df[df['UTC_Time'] == '30/10/2023 02:28']

,User_ID,UTC_Time,Account,Operation,Coin,Change,Remark
1657,71732789,30/10/2023 02:28,Spot,Simple Earn Flexible Interest,BNB,6.900000e-07,Binance Earn


In [382]:
df_test['Coin_x'] = df_test['Coin_x'].astype(str)
df_test['Coin_y'] = df_test['Coin_y'].astype(str)

In [383]:
df_test['Coin_y'][0] == "nan"

True

In [403]:
holdings

{'EUR': np.float64(140.117205),
 'BTC': np.float64(0.001163),
 'ETH': np.float64(0.03147),
 'DOGE': np.float64(1065.2337)}

In [319]:
#df_test['Coin_x'][line] in
df_test['Holdings'][line]

nan

In [320]:
holdings = {}